# Submit the DNA full-cell simulation

Run this notebook **once on Monday** from the QCB Delta Gateway to copy the workshop template to your bgvl directory and submit the ~14 hour GPU job.

- **Allocation:** `bgvl-delta-gpu`
- **Background reading:** [`README.md`](README.md) sections 6–12
- **VMD visualization:** README section 12 (Open OnDemand Desktop)

Open this notebook from **`SummerSchool_2026/DNA/submit_simulation.ipynb`** in the Jupyter file browser. Use the default Python kernel.

## 1. Clone the repository (skip if you already have it)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

BGVL_DNA = Path('/projects/bgvl/SummerSchool_2026/DNA')
PRELAUNCH = BGVL_DNA / 'files' / 'prelaunch_dna_workshop.sh'
WORK_DIR = Path(f"/projects/bgvl/{os.environ['USER']}")

def find_dna_root():
    """Locate SummerSchool_2026/DNA from clone or shared bgvl copy."""
    marker = Path('files') / 'prelaunch_dna_workshop.sh'
    candidates = []

    try:
        candidates.append(Path.cwd())
    except FileNotFoundError:
        pass

    candidates.extend([
        Path('/home/user/workspace/SummerSchool_2026/DNA'),
        Path('SummerSchool_2026/DNA'),
        BGVL_DNA,
    ])

    for candidate in candidates:
        try:
            root = candidate.resolve()
        except (OSError, RuntimeError):
            continue
        if (root / marker).is_file():
            return root

    if PRELAUNCH.is_file():
        return BGVL_DNA.resolve()

    raise FileNotFoundError(
        'Could not find DNA workshop files. Try:\n'
        '  !git clone https://github.com/Luthey-Schulten-Lab/SummerSchool_2026.git\n'
        'or ask an admin to sync /projects/bgvl/SummerSchool_2026/DNA/'
    )

def require_cmd(name):
    if shutil.which(name) is None:
        raise RuntimeError(f'{name} not found in PATH — sbatch may not work from this Gateway session')

for cmd in ('sbatch', 'squeue', 'bash'):
    require_cmd(cmd)

if not Path('SummerSchool_2026').is_dir() and not PRELAUNCH.is_file():
    !git clone https://github.com/Luthey-Schulten-Lab/SummerSchool_2026.git

dna_root = find_dna_root()
os.chdir(dna_root)
print('DNA module:', dna_root)
print('bgvl work dir:', WORK_DIR)
print('Can write bgvl dir:', os.access(WORK_DIR.parent, os.W_OK))

## 2. Copy the workshop template to `/projects/bgvl/$USER/`

In [ ]:
prelaunch = dna_root / 'files' / 'prelaunch_dna_workshop.sh'
if not prelaunch.is_file():
    prelaunch = PRELAUNCH
print('Using prelaunch script:', prelaunch)
!bash {prelaunch}

## 3. Check your workspace

In [ ]:
work_dir = WORK_DIR
if not os.access(work_dir.parent, os.W_OK):
    raise PermissionError(
        f'Cannot write under {work_dir.parent}. '
        'Make sure you selected allocation bgvl-delta-gpu on the Gateway.'
    )

os.chdir(work_dir)
print('Working directory:', work_dir)
print('launch_simulation.sh:', (work_dir / 'launch_simulation.sh').is_file())
print('template scripts:', list((work_dir / 'DNA_SummerSchool_2026/scripts').glob('*')))

## 4. Submit the Slurm job

The Apptainer image must exist on bgvl:
`/projects/bgvl/SummerSchool_2026/DNA/files/DNA_summer2025.sif`

In [ ]:
sif = Path('/projects/bgvl/SummerSchool_2026/DNA/files/DNA_summer2025.sif')
if not sif.is_file():
    raise FileNotFoundError(f'Missing container image: {sif}')
print('Container image OK:', sif)

!sbatch launch_simulation.sh

## 5. Monitor progress

Re-run the cells below while the job is running (~14 hours on an A100).

In [ ]:
!squeue -u $USER

In [ ]:
log = work_dir / 'DNA_tutorial.log'
if log.is_file():
    !tail -30 {log}
else:
    print(f'Log not created yet: {log}')

In [ ]:
traj = work_dir / 'DNA_SummerSchool_2026/data/summerschool.lammpstrj'
print('Trajectory ready:', traj.is_file())
if traj.is_file():
    print(f'Size: {traj.stat().st_size / 1e6:.1f} MB')
print(traj)